<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/02_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 02 — Tools as verbs

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

If Module 01 gave you the *mental model* for an agent, Module 02 is where the agent starts *doing things*. Tools are the verbs of agent programming. An agent without tools is a chatbot — it can only emit text. An agent with tools can look up data, call APIs, talk to other agents, run code. Everything interesting.

In this module you'll learn the **four flavors of tools** ADK supports, when to use each, and a short theoretical interlude on **risk-based tool design** — because a tool that reads data and a tool that deletes a database should not be in the same design category.

**What you'll build:** four small agents, one per tool flavor, plus one extra that demonstrates a confirmation gate on a destructive tool.

**Runs in:** Google Colab or local Jupyter. MCP demo needs the `mcp_servers/` folder from this repo — if you're on Colab, clone the repo first.

**Running cost:** under $0.02 on OpenRouter.

# Setup

## Install Dependencies

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 mcp==1.29.0 httpx==0.28.1 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## Clone the MCP servers folder (Colab only)

If you're running this locally, skip the next cell — you already have `mcp_servers/`. If you're on Colab, run it to pull the repo and `cd` into it.

In [2]:
import os
if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ:
    !git clone --depth=1 -q https://github.com/robertbarcik/ADK-tutorial /content/ADK-tutorial 2>/dev/null || true
    os.chdir("/content/ADK-tutorial")
    print(f"✅ Colab: working in {os.getcwd()}")
else:
    # Local: you should already be inside the repo.
    print(f"✅ Local: working in {os.getcwd()}")

✅ Local: working in /Users/robertbarcik/git-repos/ADK-tutorial/notebooks


## API Key Configuration

Same OpenRouter key as M01. If you have one set as a Colab secret under `OPENROUTER_API_KEY`, it'll be picked up automatically.

In [3]:
import os

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Tip: set OPENROUTER_API_KEY in Colab secrets (🔑 icon) or a local .env file.")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


## Import Libraries

Two lines in the next cell deserve a plain-language explanation; the rest is the same as M01.

**`nest_asyncio.apply()`** — In M01 you learned that `async` code needs an *event loop* to run in: the thing that juggles "who is waiting on what." Jupyter already runs one event loop of its own (that is why you can write `await` in a cell). Some libraries — the MCP client is one — insist on starting their *own* loop with `asyncio.run(...)`, and Python refuses to start a loop inside a loop. `nest_asyncio` is a small patch that lifts that restriction. You apply it once at the top and forget about it. In a plain `.py` script you would not need it.

**The `sys.stderr` swap** — An MCP server runs as a *separate process* (more on that in Flavor 3), and the library that launches it needs a real file handle for the error stream. Jupyter replaces `sys.stderr` with its own object that lacks that handle, so we hand it `/dev/null` instead. Pure plumbing; nothing to learn here except "this is why that odd block exists."

The four new imports — `OpenAPIToolset`, `McpToolset`, `StdioConnectionParams`, `AgentTool` — are the four tool flavors we build below.


In [4]:
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")

# Jupyter / Colab patch sys.stderr with an OutStream that lacks a usable .fileno().
# MCP's stdio subprocess spawn needs fileno(), so we swap stderr to a real fd.
# stdout is left alone, so print() still shows normally in the notebook.
try:
    sys.stderr.fileno()
except Exception:
    import os
    sys.stderr = open(os.devnull, "w")

import nest_asyncio
nest_asyncio.apply()

os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.openapi_tool import OpenAPIToolset
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.adk.tools.agent_tool import AgentTool
from mcp import StdioServerParameters
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The Tool Mental Model

Here's what's happening when your agent calls a tool, regardless of the flavor:

```
┌─────────────┐   "I need to know X"    ┌─────────────┐
│   LLM       │────────────────────────▶│    ADK      │
│   decides   │                         │  dispatch   │
└─────────────┘                         └──────┬──────┘
       ▲                                       │
       │  "X is {...}"                         ▼
       │                                 ┌─────────────┐
       └─────────────────────────────────│  your tool  │
                                         │  (any kind) │
                                         └─────────────┘
```

The model sees a **schema** (tool name, description, argument types) and decides when to call the tool and with what arguments. ADK handles the dispatch: it finds the Python code behind the schema, calls it, wraps the return value, and gives it back to the model as a tool-response event.

The four flavors of tools differ only in **where the schema comes from** and **where the code lives**:

| Flavor | Schema source | Code lives in |
|---|---|---|
| **FunctionTool** | Docstring + type hints of a Python function | The notebook itself |
| **OpenAPIToolset** | An OpenAPI spec | A remote HTTP API |
| **McpToolset** | The MCP server's `list_tools` response | A separate process (MCP server) |
| **AgentTool** | Another agent's `name` and `description` | That other agent |

Four flavors, same model-facing abstraction. Let's build one of each.

# Helper: Running an Agent and Printing Events

The same `chat()` helper from M01 (session → runner → `Content` → loop over events), with two small additions: an optional `app` name, and truncation of long outputs — MCP tool responses can be several hundred characters of JSON, which would swamp the notebook.

If the `async def` / `await` / `async for` pattern still looks foreign, re-read the *Running the Agent* section of M01 once more; nothing new is happening here.


In [5]:
APP = "m02_tools"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, app: str = APP):
    sid = f"s-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=app, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=app, session_service=session_service)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    print(f"USER: {prompt}\n")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        tag = "[FINAL]" if event.is_final_response() else "[step]"
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    print(f"{tag} {event.author}: {p.text.strip()[:400]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[tool_call] {p.function_call.name}({args})")
                if p.function_response:
                    resp = str(p.function_response.response)
                    # Truncate long MCP responses
                    print(f"[tool_resp] {resp[:350]}{'...' if len(resp) > 350 else ''}")

print("✅ chat() helper ready.")

✅ chat() helper ready.


# Flavor 1 — FunctionTool: Plain Python Functions

You saw this one in M01. Here's the deep version: **the docstring is the schema the model sees.** ADK reads your function's docstring + type hints and turns them into the JSON schema the LLM provider expects. Three practical consequences:

1. **Write the docstring for the model, not for a human reviewer.** The model does not have a Slack channel to ask you what a parameter means. Every argument needs an unambiguous description.
2. **Type hints are load-bearing.** `city: str` becomes `"type": "string"`. A missing type hint degrades to `"string"` or worse. Always type your tool functions.
3. **Return a JSON-serializable dict or string.** Not a custom class, not a NumPy array. The return value is sent verbatim to the model as a tool-response event.

Two bits of Python convention the example below relies on, in case they are new:

- **`units: str = "celsius"`** — a parameter with a *default value*. Python treats it as optional; ADK translates that into "optional" in the schema, so the model may leave it out and the default applies.
- **The `Args:` section** in the docstring — this is the *Google docstring style*: a line `Args:` followed by one indented line per parameter, `name: description`. ADK (as of 2.7) does not split it up: the *whole* docstring, `Args:` included, is sent to the model as the tool description, and the schema itself only carries the names, types and which parameters are optional. So the per-argument descriptions still reach the model — as text — and the `Args:` layout is simply the clearest way to write them. Run `FunctionTool(get_weather)._get_declaration()` after the cell if you want to see exactly what the model receives.

Here's a richer version of the weather tool — two parameters, one optional — that demonstrates what good tool docstrings look like.


In [6]:
def get_weather(city: str, units: str = "celsius") -> dict:
    """Look up today's weather for a city.

    Use this tool whenever the user asks about current weather. The input is a
    city name as a string; the output is a dict with 'city', 'temperature',
    'condition', and 'units'.

    Args:
        city: The city name, in English (e.g. "Prague", "Munich", "Bratislava").
        units: Either "celsius" (default) or "fahrenheit". Use "fahrenheit" only
               if the user explicitly asks for it.
    """
    fake_db = {
        "Bratislava": ("Sunny", 18),
        "Prague": ("Cloudy", 14),
        "Munich": ("Rainy", 11),
    }
    condition, celsius = fake_db.get(city, ("Unknown", None))
    if celsius is None:
        return {"city": city, "condition": "Unknown", "error": f"No data for {city}."}
    temp = celsius if units == "celsius" else round(celsius * 9/5 + 32, 1)
    return {"city": city, "temperature": temp, "condition": condition, "units": units}

weather_agent = LlmAgent(
    name="weather_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports today's weather for a city.",
    instruction=(
        "You report weather. Always use the get_weather tool. Be brief — "
        "a single short sentence."
    ),
    tools=[get_weather],
)

await chat(weather_agent, "How's the weather in Munich today, in Fahrenheit?")

USER: How's the weather in Munich today, in Fahrenheit?



[tool_call] get_weather({'city': 'Munich', 'units': 'fahrenheit'})
[tool_resp] {'city': 'Munich', 'temperature': 51.8, 'condition': 'Rainy', 'units': 'fahrenheit'}


[FINAL] weather_agent: Munich is 51.8°F and rainy today.


Two things worth noticing in that event stream:

- The model picked `units='fahrenheit'` because you asked for it in English. It did *not* hallucinate a unit the schema doesn't allow; the docstring's explicit mention of "celsius" and "fahrenheit" narrowed the choice.
- The return value is passed through as a dict. The model read the numeric `temperature` field and composed a natural-language response around it. The tool didn't need to return prose — structured data is enough.

FunctionTool is the default flavor. Reach for it unless you have a specific reason to use one of the others.

# Flavor 2 — OpenAPIToolset: Consume an Entire API

When the thing you want to call is a REST API, you probably don't want to hand-write N `FunctionTool`s for every endpoint. `OpenAPIToolset` takes a full OpenAPI spec and turns every operation into a tool, automatically.

**What is an OpenAPI spec?** A machine-readable menu of an HTTP API: for each URL path, which method (GET/POST), which parameters, what the response looks like. It is written in JSON or YAML; many APIs publish theirs at a URL like `/openapi.json`, and most web frameworks (FastAPI, for example) generate it for free. Below we write a tiny one by hand as a Python dict — read it top to bottom once: *servers* (base URL) → *paths* (`/latest`) → *get* → *parameters* (`base`, `symbols`) → *responses*. Look at the `description` strings especially: they play the same role the docstring played in Flavor 1, because they are what the model sees.

The demo uses [Frankfurter](https://api.frankfurter.dev) — a free, no-auth public currency rates API. The full Frankfurter spec has several endpoints; we'll feed ADK a minimal subset that covers "convert currency X to Y."

In production, you'd point this at your company's existing API spec — the same spec your web frontend already uses — and get all those endpoints available to your agent in one line.


In [7]:
FRANKFURTER_SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Frankfurter FX", "version": "1.0.0"},
    "servers": [{"url": "https://api.frankfurter.dev/v1"}],
    "paths": {
        "/latest": {
            "get": {
                "operationId": "getLatestRate",
                "summary": "Get today's exchange rate between two currencies.",
                "parameters": [
                    {
                        "name": "base",
                        "in": "query",
                        "description": "Source currency as a 3-letter ISO code (e.g. USD, EUR, CHF).",
                        "required": False,
                        "schema": {"type": "string"},
                    },
                    {
                        "name": "symbols",
                        "in": "query",
                        "description": "Comma-separated target currency codes (e.g. 'EUR' or 'EUR,GBP').",
                        "required": False,
                        "schema": {"type": "string"},
                    },
                ],
                "responses": {
                    "200": {
                        "description": "JSON object with 'base', 'date', and 'rates' fields.",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

fx_toolset = OpenAPIToolset(spec_dict=FRANKFURTER_SPEC)

fx_agent = LlmAgent(
    name="fx_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports live foreign exchange rates.",
    instruction=(
        "You are a currency assistant. For a request like 'USD to EUR', call "
        "get_latest_rate with base='USD' and symbols='EUR'. Currency codes are "
        "3-letter ISO codes (USD, EUR, GBP, CHF, JPY). Report the numeric rate."
    ),
    tools=[fx_toolset],
)

await chat(fx_agent, "What's the current rate from CHF to JPY?")

USER: What's the current rate from CHF to JPY?



[tool_call] get_latest_rate({'base': 'CHF', 'symbols': 'JPY'})
[tool_resp] {'amount': 1.0, 'base': 'CHF', 'date': '2026-08-21', 'rates': {'JPY': 198.5}}


[FINAL] fx_agent: 1 CHF = 198.5 JPY


Notice the tool name in the event stream: `get_latest_rate`, not `getLatestRate`. ADK snake-cases operation IDs from OpenAPI specs to match typical Python conventions. If you ever wonder why your tool by a specific name isn't being called, check whether ADK renamed it.

The tool-response event shows the raw HTTP JSON body — ADK does not transform the API's output; it just ferries it back to the model, which picks out the rate. This is a feature: you get to debug your API's contract directly in the event stream.

# Flavor 3 — McpToolset: Connect to an MCP Server

Model Context Protocol is Anthropic's standard for exposing tools to agents. It was donated in December 2025 to the Agentic AI Foundation (a Linux Foundation project co-founded by Anthropic, OpenAI and Block) and is now the de facto protocol for agent-to-tool communication across the industry.

The idea in one sentence: **the tools live in a separate program, and your agent asks that program "what tools do you have?" and then "run this one."** The separate program is the *MCP server*. It can be written by someone else, in another language, and keep its own state (database connections, credentials). Your agent only speaks the protocol.

Three terms in the code, for readers who haven't dealt with processes before:

- **Subprocess** — your notebook *starts another program* (here: a second Python interpreter running `ticket_mcp_server.py`) and keeps it running in the background for as long as the toolset lives.
- **stdio** — the two programs talk by writing lines to each other's standard input/output, the same channels `print()` and `input()` use. No network, no ports; this is the simplest MCP transport. (The alternative is HTTP, for servers running on another machine.)
- **`sys.executable`** — the path of the Python that is running this notebook. We use it as the `command` so the server starts with the same interpreter and the same installed packages; a bare `"python"` might resolve to a different installation.

This repo ships three MCP servers under `mcp_servers/`:

- `ticket_mcp_server.py` — an IT support ticket database (5 tools: get, list, create, update, search)
- `knowledge_mcp_server.py` — a help-article knowledge base
- `system_monitoring_mcp_server.py` — server uptime / disk / CPU lookups

We'll use the ticket server. `McpToolset` launches it as a subprocess and discovers its tools automatically — open `mcp_servers/ticket_mcp_server.py` in a second tab while this runs; it is short, and you will recognise the same docstring-as-schema idea from Flavor 1.


In [8]:
import os
from pathlib import Path

# Resolve mcp_servers/ticket_mcp_server.py whether the notebook runs from the
# repo root (local) or from notebooks/ (nbconvert / Jupyter Lab default cwd).
def _resolve_mcp_server(script_name: str) -> str:
    for parent in [Path.cwd(), Path.cwd().parent, Path("/content/ADK-tutorial")]:
        candidate = parent / "mcp_servers" / script_name
        if candidate.exists():
            return str(candidate.resolve())
    raise FileNotFoundError(f"Could not locate mcp_servers/{script_name}")

TICKET_SERVER_PATH = _resolve_mcp_server("ticket_mcp_server.py")
print(f"✅ MCP server script: {TICKET_SERVER_PATH}")

ticket_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,  # use the same Python that's running this notebook
            args=[TICKET_SERVER_PATH],
        ),
        timeout=20,
    )
)

ticket_agent = LlmAgent(
    name="ticket_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Helps users look up and manage IT support tickets.",
    instruction=(
        "You help the IT support team. Use search_tickets to find tickets by "
        "keyword, get_ticket by ID, list_tickets to browse, update_ticket to "
        "change a status, and create_ticket for new issues. Be concise."
    ),
    tools=[ticket_toolset],
)

await chat(ticket_agent, "Find any open tickets about WiFi.")

✅ MCP server script: /Users/robertbarcik/git-repos/ADK-tutorial/mcp_servers/ticket_mcp_server.py
USER: Find any open tickets about WiFi.



[step] ticket_agent: **Considering ticket status**

I'm thinking about how to find open tickets on WiFi. Perhaps I don't need a specific keyword since the user asked for any open tickets. I could search for "WiFi" and list open issues in parallel, but I need to intersect those results. So, my approach would be to search using "WiFi" while ensuring I filter to only show tickets with an open status. That way, I can prov
[tool_call] search_tickets({'query': 'WiFi'})
[tool_call] list_tickets({'status': 'open', 'priority': 'low', 'assigned_to': ''})
[tool_resp] {'content': [{'type': 'text', 'text': '{\n  "query": "WiFi",\n  "count": 1,\n  "tickets": [\n    {\n      "id": "T-1003",\n      "title": "WiFi connectivity issues",\n      "description": "WiFi keeps disconnecting every 10 minutes in Conference Room B",\n      "status": "in_progress",\n      "priority": "high",\n      "assigned_to": "network_team",...
[tool_resp] {'content': [{'type': 'text', 'text': '{\n  "count": 1,\n  "tickets": [

[FINAL] ticket_agent: No open WiFi tickets found. The only WiFi ticket is **T-1003 — “WiFi connectivity issues”**, currently **in progress** and assigned to `network_team`.


What just happened under the hood: `McpToolset` started `ticket_mcp_server.py` as a subprocess, spoke the MCP handshake over stdio, and asked it for the list of tools. The server returned a schema for five tools. ADK registered those as if they were local FunctionTools, and from the agent's perspective there's no difference — it just sees `search_tickets` and calls it.

Two details in the stream that are new compared with M01: a `[step]` text event *before* the tool calls — that is GPT-5.6's short reasoning summary, which OpenAI models emit as ordinary text (other vendors don't; M04 makes a point of these differences) — and **two tool calls in one event**. The model asked for `search_tickets` and `list_tickets` at the same time; ADK ran both and returned both results before the model answered. Parallel tool calls are a model capability, not something you configure.

**This is the most valuable part of the flavor.** Your MCP server can be written in any language (TypeScript, Go, Rust), run on any machine, and own any state — database connections, API credentials, caches. The agent doesn't care. It talks the protocol, the tools just work.

When you're done with an MCP toolset, close it to shut down the subprocess cleanly. We'll do that at the end of the notebook.

# Flavor 4 — AgentTool: Wrap an Agent as a Tool

The fourth flavor is the most conceptually interesting, and the one Module 06 will unpack fully. For now, the one-sentence version: **`AgentTool` takes any existing agent and makes it callable as a tool from another agent.**

Why this is useful: sometimes you have a specialist agent — a translator, a data-lookup agent, a math reasoning agent — and you want a parent orchestrator to *call* it rather than hand control over to it. The parent stays in charge; the specialist answers one question and hands control back. If that sounds like a function call, that's the mental model.

The contrast is with `sub_agents` — the parent's LLM can decide to *transfer* to a child agent, which takes over the conversation until it decides to transfer back. That's delegation, not a function call. We'll compare the two in M06. For now, focus on the `AgentTool` pattern:

In [9]:
# A specialist agent that translates short phrases to Slovak.
translator = LlmAgent(
    name="translator",
    model=LiteLlm(model=MODEL_STRING),
    description="Translates a short English phrase into Slovak. Input: the phrase. Output: the Slovak translation.",
    instruction=(
        "You are a translation tool. Input is an English phrase. Output is the "
        "Slovak translation only — no commentary, no prefix, no quotes. Respond "
        "with the translation and nothing else."
    ),
)

# Wrap the translator as a callable tool, then give it to a parent agent.
orchestrator = LlmAgent(
    name="orchestrator",
    model=LiteLlm(model=MODEL_STRING),
    description="A helpful assistant that can translate phrases via a specialist.",
    instruction=(
        "You answer questions. When the user asks for a translation to Slovak, "
        "call the translator tool with the phrase to translate. Include the "
        "translator's result in your reply."
    ),
    tools=[AgentTool(agent=translator)],
)

await chat(orchestrator, "How do you say 'good morning, my friend' in Slovak?")

USER: How do you say 'good morning, my friend' in Slovak?



[tool_call] translator({'request': 'good morning, my friend'})


[tool_resp] {'result': 'Dobré ráno, môj priateľu'}


[FINAL] orchestrator: Dobré ráno, môj priateľu.


The event stream shows the parent agent calling `translator` as if it were a function. The specialist ran in its own context (a fresh LLM call), produced its translation, and the parent wrapped the result into a natural reply.

`AgentTool` is the right answer when:
- The child agent has a clean input/output contract (like a function).
- You want the parent to stay in charge of the conversation.
- You want the child's work to be visible in the parent's event stream (it is — you can see the call and the response).

Use `sub_agents` instead when the child should own the conversation for a stretch of turns. M06 will make the distinction crisp with a side-by-side demo.

# Interlude — Risk-based Tool Design

> *From the "Agentic Design Patterns" publication, Pattern 4. This 2-minute interlude is theory; the code below is the minimal demonstration.*

Not all tools are equal. A tool that reads a ticket database is not the same category as a tool that deletes the database. The difference is **blast radius** — the scope of damage a misfiring tool call can do before anyone notices.

A practical taxonomy:

| Risk tier | Characteristic | Example | Guard |
|---|---|---|---|
| **Read-only** | Doesn't change anything external | `get_weather`, `search_tickets` | None needed |
| **Mutating, reversible** | Writes, but the write is easy to undo | `create_ticket`, `send_draft_email` | Log every call |
| **Mutating, irreversible** | Writes that can't be easily rolled back | `charge_credit_card`, `post_to_slack`, `execute_sql` | **Explicit confirmation** |
| **Catastrophic** | Destructive, multi-user, loud | `drop_database`, `delete_user`, `publish_press_release` | **Human in the loop.** Do not let the agent call these directly. |

The temptation is to treat every tool the same because the agent framework treats them the same. Don't. Put the guards in the *tool implementation*, not in the instruction — an instruction is a polite request the model can ignore. A code-level guard is a wall.

Here's the minimal pattern: a `delete_ticket` tool that refuses to execute without a confirmation token.

In [10]:
# Simulated ticket DB so we can "delete" from it.
TICKETS = {"T-1001": "Laptop won't boot", "T-1002": "Password reset request"}

def delete_ticket(ticket_id: str, confirmation_token: str = "") -> dict:
    """Delete a support ticket. IRREVERSIBLE.

    Args:
        ticket_id: The ticket to delete, e.g. "T-1001".
        confirmation_token: MUST equal "CONFIRM_DELETE_<ticket_id>" for the
            delete to proceed. If empty or wrong, returns a preview and does
            nothing. The human confirms by providing the token explicitly.
    """
    expected = f"CONFIRM_DELETE_{ticket_id}"
    if confirmation_token != expected:
        return {
            "status": "preview",
            "ticket_id": ticket_id,
            "current_title": TICKETS.get(ticket_id, "<unknown>"),
            "message": (
                f"Delete not executed. To proceed, pass confirmation_token="
                f"'{expected}'. Ask the user to confirm before retrying."
            ),
        }
    if ticket_id not in TICKETS:
        return {"status": "error", "message": f"Ticket {ticket_id} not found."}
    title = TICKETS.pop(ticket_id)
    return {"status": "deleted", "ticket_id": ticket_id, "former_title": title}

guarded_agent = LlmAgent(
    name="guarded_ticket_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Manages tickets, with a confirmation gate on destructive operations.",
    instruction=(
        "You manage tickets. For delete_ticket: never call with "
        "confirmation_token on the first attempt. Always call it with an empty "
        "token first to preview, show the user the preview, and only call with "
        "the correct token if the user explicitly confirms."
    ),
    tools=[delete_ticket],
)

await chat(guarded_agent, "Delete ticket T-1001.")

USER: Delete ticket T-1001.



[tool_call] delete_ticket({'ticket_id': 'T-1001', 'confirmation_token': ''})
[tool_resp] {'status': 'preview', 'ticket_id': 'T-1001', 'current_title': "Laptop won't boot", 'message': "Delete not executed. To proceed, pass confirmation_token='CONFIRM_DELETE_T-1001'. Ask the user to confirm before retrying."}


[FINAL] guarded_ticket_agent: Ticket **T-1001** (“Laptop won't boot”) is ready for deletion, but it has **not been deleted**.

Please explicitly confirm the deletion, for example: **“Confirm deletion of T-1001.”**


Read the event stream. The agent called `delete_ticket` without a confirmation token, got back a **preview** response, and surfaced it to the user — rather than silently destroying the ticket. The actual delete only happens if a second user turn provides the confirmation.

The key design move: **the guard is in the tool's code**, enforced by a string-equality check. The instruction tells the model how to behave politely, but the tool *cannot be tricked* into executing without the token, no matter what the model does. This is the pattern to carry forward for any tool with irreversible consequences.

## Cleanup

Close the MCP toolset to shut down the ticket-server subprocess cleanly. Omit this step and you'll leak a background Python process for every notebook restart. (`close()` is `async` like everything else in ADK, hence the `await`.)


In [11]:
await ticket_toolset.close()
print("✅ MCP subprocess closed.")

✅ MCP subprocess closed.


# Your Turn

Four small tasks, one per flavor plus the interlude. Each should take about five minutes in its own new cell below.

1. **FunctionTool.** Extend `get_weather` to take a third optional argument `days_ahead: int = 0` and have the function return a different (fake) forecast for each day. Ask the agent for tomorrow's weather in Prague. Does the model call the tool with `days_ahead=1`?
2. **OpenAPIToolset.** Add a second path to `FRANKFURTER_SPEC` — `/currencies` (which returns a JSON dict of all supported currency codes). Ask the agent *"What currencies do you support?"* — does it call the new operation?
3. **McpToolset.** Swap `ticket_mcp_server.py` for `knowledge_mcp_server.py` in `McpToolset`. Build a new agent that searches the knowledge base. What tools does it expose?
4. **AgentTool.** Wrap the `fx_agent` (from flavor 2) as an `AgentTool` and add it to the `orchestrator`. Ask the orchestrator *"Translate 'Today 1 USD is X EUR' into Slovak, with X filled from the live rate."* — does it call both tools in one turn?
5. **Risk-based.** Add a second destructive tool, `wipe_all_tickets()`, that returns a preview with a counter like "this would delete 17 tickets" and requires the token `"CONFIRM_WIPE_ALL"`. Verify the preview comes back correctly.

# Key Takeaways

- Tools are the verbs of an agent. Without them, you have a chatbot.
- **FunctionTool**: plain Python functions. Docstring + type hints ARE the schema. Default choice.
- **OpenAPIToolset**: one line, any REST API with a spec, N tools for free.
- **McpToolset**: connect to a separate tool-server process (any language, any state) over the MCP protocol.
- **AgentTool**: wrap a specialist agent so its parent can call it like a function. Keeps the parent in charge.
- **Risk-based tool design**: blast radius matters. Put confirmation gates *in the tool code*, not only the instruction. An instruction is a request; code is a wall.

# Next up — M03: Sessions, State, Events, Artifacts

Every tool call you just saw went into the session's event history. M03 makes sessions the subject: how to persist them, how to put state in them, how to scope state with prefixes (`user:`, `app:`, `temp:`), and why the boundary between *session state* and *long-term memory* matters for production agents. See you there.